# AI-Based Predictive Farming System — Tomato Disease Detection Model Training
This notebook trains an independent **MobileNetV2** transfer learning model for **Tomato** disease classification.

### Strict Constraints:
- Separate model and separate dataset per plant (no shared weights or multi-crop mixing).
- Transfer learning with MobileNetV2 (ImageNet weights).
- Preprocessing: 224x224 RGB, normalized to [0.0, 1.0].
- Train/Validation/Test split with data augmentation.
- Early stopping and model checkpointing.
- Evaluation: Accuracy, Precision, Recall, F1 Score, Classification Report, Confusion Matrix.


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

os.environ['KERAS_BACKEND'] = 'torch'
import keras
from keras import layers
from keras.applications import MobileNetV2
print('Keras version:', keras.__version__)


In [ ]:
# Configuration
PLANT_ID = 'tomato'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
DATASET_DIR = r'C:\Users\CreoDhare\Downloads\plantdataset\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)'
OUTPUT_MODEL_DIR = Path('../../disease_models') / PLANT_ID
OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
CLASSES = [
  "Bacterial_spot",
  "Early_blight",
  "Late_blight",
  "Leaf_Mold",
  "Septoria_leaf_spot",
  "Spider_mites Two-spotted_spider_mite",
  "Target_Spot",
  "Tomato_Yellow_Leaf_Curl_Virus",
  "Tomato_mosaic_virus",
  "Healthy"
]


## 1. Dataset Loading and Exploration
Loading train and validation datasets with image preprocessing and augmentation.

In [ ]:
# Load datasets using Keras utils
# train_ds = keras.utils.image_dataset_from_directory(
#     os.path.join(DATASET_DIR, 'train'),
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     label_mode='categorical'
# )
# val_ds = keras.utils.image_dataset_from_directory(
#     os.path.join(DATASET_DIR, 'valid'),
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     label_mode='categorical'
# )


## 2. Model Architecture (MobileNetV2 Transfer Learning)
MobileNetV2 base pretrained on ImageNet, frozen initial base layers, global average pooling, dropout, and Softmax dense head.

In [ ]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
# Augmentation pipeline
x = layers.RandomFlip('horizontal')(inputs)
x = layers.RandomRotation(0.1)(x)
x = layers.RandomZoom(0.1)(x)
# Feature extraction
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(10, activation='softmax')(x)
model = keras.Model(inputs, outputs, name='tomato_disease_mobilenetv2')
model.summary()


## 3. Training with Early Stopping & Model Checkpoint


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(str(OUTPUT_MODEL_DIR / 'model.keras'), save_best_only=True)
]


## 4. Evaluation and Independent Test Metrics
Classification report, precision, recall, F1-score, and confusion matrix.

In [ ]:
# Evaluate on validation / independent test split
# Generate evaluation_report.json and confusion matrix
